# LA Crime Data — KPI Dashboard (Native Notebook Version)

Interactive EDA dashboard built with **pandas** + **Plotly** + **ipywidgets**.
Pick a year from the dropdown below and every KPI and chart updates in place.

> If this is your first time running the notebook, make sure the packages below are installed:
> ```
> pip install pandas plotly ipywidgets
> ```
> If you're on classic Jupyter Notebook (not JupyterLab), you may also need:
> ```
> jupyter nbextension enable --py widgetsnbextension
> ```


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

pd.options.mode.chained_assignment = None

## 1. Load data

Adjust `CSV_PATH` if your file lives somewhere else.

In [2]:
CSV_PATH = "LA_Crime_Data_Cleaned.csv"  # <-- update this path if needed

COLS = [
    'AREA NAME', 'Crm Cd Desc', 'Vict Age', 'Vict Sex Label', 'Vict Descent Label',
    'Weapon Desc Clean', 'Status Desc', 'Year OCC', 'Month OCC', 'Weekday OCC',
    'Hour OCC', 'Age_Group'
]

df = pd.read_csv(CSV_PATH, usecols=COLS)
print(f"Loaded {len(df):,} records spanning {df['Year OCC'].min()}–{df['Year OCC'].max()}")
df.head()

Loaded 1,005,198 records spanning 2020–2025


,AREA NAME,Crm Cd Desc,Vict Age,Status Desc,Hour OCC,Year OCC,Month OCC,Weekday OCC,Vict Sex Label,Vict Descent Label,Weapon Desc Clean,Age_Group
0,Wilshire,Vehicle - Stolen,0,Adult Arrest,21,2020,3,Sunday,Male,Other,No Weapon Involved,NaN
1,Central,Burglary From Vehicle,47,Invest Cont,18,2020,2,Saturday,Male,Other,No Weapon Involved,46-60
2,Southwest,Bike - Stolen,19,Invest Cont,17,2020,11,Wednesday,Unknown,Unknown,No Weapon Involved,19-30
3,Van Nuys,Shoplifting-Grand Theft ($950.01 & Over),19,Invest Cont,20,2020,3,Tuesday,Male,Other,No Weapon Involved,19-30
4,Hollenbeck,Vehicle - Stolen,0,Invest Cont,6,2020,9,Wednesday,Unknown,Unknown,No Weapon Involved,NaN


## 2. Build per-year KPI summaries

This mirrors the logic from the HTML dashboard, computed natively in pandas.

In [3]:
WEEKDAY_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
MONTH_NAMES   = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
AGE_ORDER     = ['0-18','19-30','31-45','46-60','60+']

def build_summary(sub: pd.DataFrame) -> dict:
    n = len(sub)
    d = {'total': n}

    status_counts = sub['Status Desc'].value_counts()
    arrest_mask = [s for s in status_counts.index if 'Arrest' in s]
    arrests = status_counts[arrest_mask].sum() if arrest_mask else 0
    d['arrest_rate'] = round(100 * arrests / n, 2) if n else 0.0
    d['status'] = status_counts

    valid_age = sub.loc[sub['Vict Age'] > 0, 'Vict Age']
    d['avg_age'] = round(valid_age.mean(), 1) if len(valid_age) else 0.0

    d['crimes'] = sub['Crm Cd Desc'].value_counts().head(12)
    d['top_crime'] = d['crimes'].index[0] if len(d['crimes']) else '—'

    d['area'] = sub['AREA NAME'].value_counts()
    d['top_area'] = d['area'].index[0] if len(d['area']) else '—'

    d['month'] = sub['Month OCC'].value_counts().sort_index()
    d['weekday'] = sub['Weekday OCC'].value_counts().reindex(WEEKDAY_ORDER).fillna(0)
    d['hour'] = sub['Hour OCC'].value_counts().sort_index()

    d['sex'] = sub['Vict Sex Label'].value_counts()
    d['descent'] = sub['Vict Descent Label'].value_counts().head(6)
    d['age_group'] = sub['Age_Group'].value_counts().reindex(AGE_ORDER).dropna()

    d['weapon'] = sub['Weapon Desc Clean'].value_counts()
    no_weapon = d['weapon'].get('No Weapon Involved', 0)
    d['pct_no_weapon'] = round(100 * no_weapon / n, 2) if n else 0.0

    return d

YEARS = sorted(df['Year OCC'].unique().tolist())
BY_YEAR = {str(y): build_summary(df[df['Year OCC'] == y]) for y in YEARS}
BY_YEAR['All'] = build_summary(df)

print("Years available:", YEARS)

Years available: [2020, 2021, 2022, 2023, 2024, 2025]


## 3. KPI card renderer

In [4]:
def kpi_html(year: str, d: dict) -> str:
    label = f"{YEARS[0]}–{YEARS[-1]}" if year == 'All' else year
    arrests = d['status'].get('Adult Arrest', 0) + d['status'].get('Juv Arrest', 0)
    top_crime_share = (100 * d['crimes'].get(d['top_crime'], 0) / d['total']) if d['total'] else 0

    cards = [
        ("Total Reported Incidents" if year == 'All' else f"Incidents in {year}", f"{d['total']:,}", f"{label} \u00b7 {d['total']:,} records"),
        ("Arrest Rate", f"{d['arrest_rate']}%", f"{arrests:,} arrests logged"),
        ("Most Reported Crime", d['top_crime'], f"{top_crime_share:.1f}% of incidents"),
        ("Highest-Volume Division", d['top_area'], f"{d['area'].get(d['top_area'], 0):,} incidents"),
        ("Avg. Victim Age", f"{d['avg_age']} yrs", f"{d['pct_no_weapon']}% incidents unarmed"),
    ]

    style = """
    <style>
      .kpi-wrap{display:flex;gap:12px;flex-wrap:wrap;font-family:-apple-system,Segoe UI,Roboto,sans-serif;margin-bottom:8px;}
      .kpi-card{background:#161b21;border:1px solid rgba(255,255,255,0.08);border-radius:12px;
                padding:14px 16px;min-width:170px;flex:1;color:#eef1f4;}
      .kpi-card .l{font-size:10.5px;text-transform:uppercase;letter-spacing:.10em;color:#8b93a1;font-weight:700;}
      .kpi-card .v{font-size:20px;font-weight:800;margin-top:6px;}
      .kpi-card .f{font-size:11.5px;color:#8b93a1;margin-top:4px;}
    </style>
    """
    body = "".join(
        f'<div class="kpi-card"><div class="l">{lab}</div><div class="v">{val}</div><div class="f">{foot}</div></div>'
        for lab, val, foot in cards
    )
    return style + f'<div class="kpi-wrap">{body}</div>'

## 4. Chart builder

Builds the full grid of Plotly figures for a given year's summary.

In [5]:
TEMPLATE = "plotly_dark"
COLORS = dict(accent="#ff5a36", accent2="#2fb4c9", accent3="#f2b705", good="#3ecf8e", purple="#9b7bff")

def build_figures(year: str, d: dict):
    figs = []

    # Yearly trend (always full range, current year highlighted)
    totals = [BY_YEAR[str(y)]['total'] for y in YEARS]
    bar_colors = [COLORS['accent'] if (year != 'All' and str(y) == year) else 'rgba(255,90,54,0.35)' for y in YEARS]
    fig = go.Figure(go.Bar(x=[str(y) for y in YEARS], y=totals, marker_color=bar_colors))
    fig.update_layout(template=TEMPLATE, title="Reported Incidents by Year", height=320, margin=dict(t=40,b=20))
    figs.append(fig)

    # Top crimes
    fig = px.bar(d['crimes'].sort_values(), orientation='h', template=TEMPLATE,
                 labels={'value':'incidents','index':''}, title="Top Crime Types")
    fig.update_traces(marker_color=COLORS['accent2'])
    fig.update_layout(height=340, showlegend=False, margin=dict(t=40,b=20))
    figs.append(fig)

    # Hour of day
    fig = px.bar(x=d['hour'].index.astype(str)+":00", y=d['hour'].values, template=TEMPLATE,
                 labels={'x':'hour','y':'incidents'}, title="Incidents by Hour of Day")
    fig.update_traces(marker_color=COLORS['accent3'])
    fig.update_layout(height=300, showlegend=False, margin=dict(t=40,b=20))
    figs.append(fig)

    # Weekday
    fig = px.bar(x=d['weekday'].index, y=d['weekday'].values, template=TEMPLATE,
                 labels={'x':'','y':'incidents'}, title="Incidents by Day of Week")
    fig.update_traces(marker_color=COLORS['good'])
    fig.update_layout(height=300, showlegend=False, margin=dict(t=40,b=20))
    figs.append(fig)

    # Monthly seasonality
    month_x = [MONTH_NAMES[m-1] for m in d['month'].index]
    fig = px.line(x=month_x, y=d['month'].values, template=TEMPLATE, markers=True,
                   labels={'x':'','y':'incidents'}, title="Seasonality by Month")
    fig.update_traces(line_color=COLORS['purple'])
    fig.update_layout(height=300, showlegend=False, margin=dict(t=40,b=20))
    figs.append(fig)

    # Area breakdown
    fig = px.bar(d['area'].sort_values(), orientation='h', template=TEMPLATE,
                 labels={'value':'incidents','index':''}, title="Incidents by Patrol Division")
    fig.update_traces(marker_color=COLORS['accent'])
    fig.update_layout(height=460, showlegend=False, margin=dict(t=40,b=20))
    figs.append(fig)

    # Case status
    fig = px.pie(names=d['status'].index, values=d['status'].values, template=TEMPLATE,
                 title="Case Status Breakdown", hole=0.55)
    fig.update_layout(height=340, margin=dict(t=40,b=20))
    figs.append(fig)

    # Victim sex
    fig = px.pie(names=d['sex'].index, values=d['sex'].values, template=TEMPLATE,
                 title="Victim Sex", hole=0.5)
    fig.update_layout(height=300, margin=dict(t=40,b=20))
    figs.append(fig)

    # Age group
    fig = px.bar(x=d['age_group'].index, y=d['age_group'].values, template=TEMPLATE,
                 labels={'x':'','y':'victims'}, title="Victim Age Group")
    fig.update_traces(marker_color=COLORS['accent3'])
    fig.update_layout(height=300, showlegend=False, margin=dict(t=40,b=20))
    figs.append(fig)

    # Descent
    fig = px.bar(d['descent'].sort_values(), orientation='h', template=TEMPLATE,
                 labels={'value':'victims','index':''}, title="Victim Descent (Top 6)")
    fig.update_traces(marker_color=COLORS['good'])
    fig.update_layout(height=300, showlegend=False, margin=dict(t=40,b=20))
    figs.append(fig)

    # Weapon (excluding "no weapon" for scale)
    weapon_no_none = d['weapon'].drop('No Weapon Involved', errors='ignore').head(10)
    fig = px.bar(weapon_no_none.sort_values(), orientation='h', template=TEMPLATE,
                 labels={'value':'incidents','index':''}, title="Weapon Involvement (Top Categories)")
    fig.update_traces(marker_color=COLORS['purple'])
    fig.update_layout(height=380, showlegend=False, margin=dict(t=40,b=20))
    figs.append(fig)

    # Weapon presence donut
    no_weapon = d['weapon'].get('No Weapon Involved', 0)
    weapon_present = d['total'] - no_weapon
    fig = px.pie(names=['No Weapon Involved','Weapon Involved'], values=[no_weapon, weapon_present],
                 template=TEMPLATE, title="Weapon Presence", hole=0.6,
                 color_discrete_sequence=['#3a4048', COLORS['accent']])
    fig.update_layout(height=340, margin=dict(t=40,b=20))
    figs.append(fig)

    return figs

## 5. Dropdown widget + live render

Run this cell — pick a year from the dropdown and the whole dashboard updates below it.

In [6]:
year_dropdown = widgets.Dropdown(
    options=['All'] + [str(y) for y in YEARS],
    value='All',
    description='Year:',
    style={'description_width': 'initial'}
)
output = widgets.Output()

def render(change=None):
    year = year_dropdown.value
    d = BY_YEAR[year]
    with output:
        clear_output(wait=True)
        display(HTML(kpi_html(year, d)))
        for fig in build_figures(year, d):
            fig.show()

year_dropdown.observe(render, names='value')

display(year_dropdown, output)
render()

Dropdown(description='Year:', options=('All', '2020', '2021', '2022', '2023', '2024', '2025'), style=Descripti…

Output()